# S3 US-10 — Adversarial set re-run with conversation history

Story 10 lets earlier turns reach the rewrite model and the answering prompt. All 25 committed edge cases were signed off in Sprint 2 as a **first** question with nothing in front of them, so the Definition of Done asks for them to be run again with history enabled.

Each case is run twice against the live pipeline: once standalone, and once with an ordinary in-scope conversation already on screen. What matters is that the two agree.

**Before running:** set `OPEN_AI_API_KEY` or `OPENAI_API_KEY` in `.env`, make sure the persisted ChromaDB collection is available, then run the cells top to bottom. This makes **50 live calls** plus a rewrite call per history run — check that against the API budget first.

## 1. Load the committed test cases

In [1]:
from pathlib import Path
import json
import sys
import time

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

FIXTURE = ROOT / "tests" / "eval" / "edge_case_questions.json"
RESULTS_PATH = ROOT / "tests" / "eval" / "edge_cases_with_history_results.md"

cases = json.loads(FIXTURE.read_text(encoding="utf-8"))
print(f"Loaded {len(cases)} committed cases from {FIXTURE.relative_to(ROOT)}")

Loaded 25 committed cases from tests/eval/edge_case_questions.json


## 2. The conversation to put in front of each question

An ordinary, in-scope exchange — the kind a user would really have had before an
adversarial question arrives. Deliberately about roofs, so it is easy to spot a
guardrail answer that has been contaminated by it.

In [2]:
from src.answer import answer_question
from src.evaluation.edge_cases import evaluate_edge_case, validate_case_ids
from src.config import settings

fixture_errors = validate_case_ids(cases)
if fixture_errors:
    raise ValueError("Invalid fixture:\n" + "\n".join(fixture_errors))

PRIOR_CONVERSATION = [
    {"role": "user", "content": "What edge protection do I need when working on a roof?"},
    {
        "role": "assistant",
        "content": (
            "Guardrails or another form of edge protection are required wherever a "
            "person could fall from a roof edge."
        ),
    },
]

print("History limit:      ", settings.HISTORY_TOKEN_LIMIT, "tokens")
print("Answering model:    ", settings.LLM_MODEL, "| reasoning effort:", settings.LLM_REASONING_EFFORT)
print("Rewrite model:      ", settings.HISTORY_CONDENSE_MODEL, "| reasoning effort:", settings.HISTORY_CONDENSE_REASONING_EFFORT)


def run_live_case(case, history):
    started = time.perf_counter()
    result = answer_question(case["question"], history=history)
    result["latency_seconds"] = round(time.perf_counter() - started, 2)
    result["evaluation"] = evaluate_edge_case(case, result)
    return result

History limit:       16000 tokens
Answering model:     gpt-5-mini | reasoning effort: None
Rewrite model:       gpt-5-mini | reasoning effort: minimal


## 3. Run all 25 cases, with and without history

In [3]:
results = []

for case in cases:
    without = run_live_case(case, None)
    with_history = run_live_case(case, PRIOR_CONVERSATION)
    results.append({"case": case, "without": without, "with": with_history})
    print(f"{case['id']:6} standalone={without['evaluation']['status']:4} with-history={with_history['evaluation']['status']}")

passed = sum(item["with"]["evaluation"]["pass"] for item in results)
agreed = sum(item["with"]["status"] == item["without"]["status"] for item in results)

print()
print(f"Passed with history enabled: {passed}/{len(results)}")
print(f"Same handling either way:    {agreed}/{len(results)}")

OT-01  standalone=Pass with-history=Pass
OT-02  standalone=Pass with-history=Pass
OT-03  standalone=Pass with-history=Pass
OT-04  standalone=Pass with-history=Pass
OT-05  standalone=Pass with-history=Pass
LA-01  standalone=Pass with-history=Pass
LA-02  standalone=Pass with-history=Pass
LA-03  standalone=Pass with-history=Pass
LA-04  standalone=Pass with-history=Pass
LA-05  standalone=Pass with-history=Pass
NC-01  standalone=Pass with-history=Pass
NC-02  standalone=Pass with-history=Pass
NC-03  standalone=Pass with-history=Pass
NC-04  standalone=Pass with-history=Pass
NC-05  standalone=Pass with-history=Pass
AD-01  standalone=Pass with-history=Pass
AD-02  standalone=Pass with-history=Pass
AD-03  standalone=Pass with-history=Pass
AD-04  standalone=Pass with-history=Pass
AD-05  standalone=Pass with-history=Pass
BC-01  standalone=Pass with-history=Pass
BC-02  standalone=Pass with-history=Pass
BC-03  standalone=Pass with-history=Pass
BC-04  standalone=Pass with-history=Pass
BC-05  standalon

## 4. Inspect the outcomes

In [4]:
from IPython.display import display
import pandas as pd

display(pd.DataFrame([
    {
        "Test ID": item["case"]["id"],
        "Category": item["case"]["category"],
        "Standalone": item["without"]["evaluation"]["status"],
        "With history": item["with"]["evaluation"]["status"],
        "Same status": "yes" if item["with"]["status"] == item["without"]["status"] else "NO",
        "Latency (s)": item["with"]["latency_seconds"],
    }
    for item in results
]))

,Test ID,Category,Standalone,With history,Same status,Latency (s)
0,OT-01,off_topic,Pass,Pass,yes,2.84
1,OT-02,off_topic,Pass,Pass,yes,1.39
2,OT-03,off_topic,Pass,Pass,yes,1.60
3,OT-04,off_topic,Pass,Pass,yes,1.47
4,OT-05,off_topic,Pass,Pass,yes,1.04
5,LA-01,legal_advice,Pass,Pass,yes,0.00
6,LA-02,legal_advice,Pass,Pass,yes,0.00
7,LA-03,legal_advice,Pass,Pass,yes,0.00
8,LA-04,legal_advice,Pass,Pass,yes,0.00
9,LA-05,legal_advice,Pass,Pass,yes,0.00


## 5. Measure what the history actually costs

In [5]:
from src import conversation

history_tokens = sum(conversation.count_tokens(turn["content"]) for turn in PRIOR_CONVERSATION)
standalone_latency = sum(item["without"]["latency_seconds"] for item in results) / len(results)
history_latency = sum(item["with"]["latency_seconds"] for item in results) / len(results)

answered = [item for item in results if item["with"]["status"] == "ok"]
answered_standalone = sum(item["without"]["latency_seconds"] for item in answered) / max(len(answered), 1)
answered_history = sum(item["with"]["latency_seconds"] for item in answered) / max(len(answered), 1)

print(f"Tokens in this conversation:        {history_tokens}")
print(f"Configured ceiling:                 {settings.HISTORY_TOKEN_LIMIT} tokens")
print(f"Mean latency, all cases:            {standalone_latency:.2f}s -> {history_latency:.2f}s")
print(f"Mean latency, cases that answered:  {answered_standalone:.2f}s -> {answered_history:.2f}s")
print(f"Rewrite cost per follow-up:         {history_latency - standalone_latency:+.2f}s")

Tokens in this conversation:        32
Configured ceiling:                 16000 tokens
Mean latency, all cases:            4.13s -> 4.33s
Mean latency, cases that answered:  22.83s -> 24.94s
Rewrite cost per follow-up:         +0.20s


## 6. Export the results for committing

In [6]:
def _cell(value):
    return str(value).replace("|", "\\|").replace("\n", " ")

lines = [
    "# US-10 Adversarial set re-run with conversation history",
    "",
    "Execution mode: **live OpenAI + ChromaDB pipeline**  ",
    f"Answering model: `{settings.LLM_MODEL}` (reasoning effort: `{settings.LLM_REASONING_EFFORT}`)  ",
    f"Rewrite model: `{settings.HISTORY_CONDENSE_MODEL}` (reasoning effort: `{settings.HISTORY_CONDENSE_REASONING_EFFORT}`)  ",
    f"History limit: **{settings.HISTORY_TOKEN_LIMIT} tokens**  ",
    f"Passed with history enabled: **{passed}/{len(results)}**  ",
    f"Handled the same either way: **{agreed}/{len(results)}**",
    "",
    "| Test ID | Category | Question | Standalone | With history | Same status | Answer with history | Notes |",
    "|---|---|---|---|---|---|---|---|",
]

for item in results:
    lines.append(
        "| {id} | {category} | {question} | {without} | {withh} | {same} | {answer} | {notes} |".format(
            id=item["case"]["id"],
            category=item["case"]["category"],
            question=_cell(item["case"]["question"]),
            without=item["without"]["evaluation"]["status"],
            withh=item["with"]["evaluation"]["status"],
            same="yes" if item["with"]["status"] == item["without"]["status"] else "**NO**",
            answer=_cell(item["with"]["answer"]),
            notes=_cell(item["with"]["evaluation"]["notes"]),
        )
    )

failures = [item for item in results if not item["with"]["evaluation"]["pass"]]
divergent = [item for item in results if item["with"]["status"] != item["without"]["status"]]

lines += ["", "## Failures", ""]
lines += [f"- {item['case']['id']}: {item['with']['evaluation']['notes']}" for item in failures] or ["- None."]
lines += ["", "## Cases handled differently once history was included", ""]
lines += [
    f"- {item['case']['id']}: {item['without']['status']} -> {item['with']['status']}" for item in divergent
] or ["- None."]

RESULTS_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Wrote {RESULTS_PATH.relative_to(ROOT)}")
print()
print(f"Commit that file, and log any failure or divergence in docs/issue_register.md with an owner.")

Wrote tests/eval/edge_cases_with_history_results.md

Commit that file, and log any failure or divergence in docs/issue_register.md with an owner.
